In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import re
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dense, Dropout, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

2025-09-23 23:12:59.451109: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-23 23:12:59.546487: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-23 23:12:59.573601: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-23 23:13:01.254293: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
df_data = pd.read_json("Entity Recognition in Resumes.json", lines=True)
df_data = df_data.drop(['extras'], axis=1)

In [3]:
df_data.head()

,content,annotation
0,Abhishek Jha\nApplication Development Associat...,"[{'label': ['Skills'], 'points': [{'start': 12..."
1,Afreen Jamadar\nActive member of IIIT Committe...,"[{'label': ['Email Address'], 'points': [{'sta..."
2,"Akhil Yadav Polemaina\nHyderabad, Telangana - ...","[{'label': ['Skills'], 'points': [{'start': 37..."
3,Alok Khandai\nOperational Analyst (SQL DBA) En...,"[{'label': ['Skills'], 'points': [{'start': 80..."
4,Ananya Chavan\nlecturer - oracle tutorials\n\n...,"[{'label': ['Degree'], 'points': [{'start': 20..."


In [4]:
df_data['content'] = df_data['content'].str.replace("\n", " ")

In [5]:
data=[]
for i in range(len(df_data)):
    enti=df_data.iloc[i]
    text=enti[0]
    entities=[]
    for i in range(len(enti[1])):
        try:
            label=enti[1][i]['label'][0]
            start_point=enti[1][i]['points'][0]['start']
            end_point=enti[1][i]['points'][0]['end']
            entities.append((start_point,end_point,label))
        #print(entities)
        #located_text=end_point=enti[1][0]['points'][0]['text']
        except:
            pass
        #print(f'Text \n {text} \n Label \n {label} \nStart Label \n {start_point} \n End Point \n {end_point}')
    #print(entities)
    data.append((text,{'entities':entities}))

In [6]:
def tokenize_and_label_nltk(text, entities):
    tokens = text.split() 
    labels = ['O'] * len(tokens)
    for start, end, label in entities:
        entity_tokens = [token for token in tokens if text.find(token) >= start and text.find(token) < end]
        if entity_tokens:
            first_token_index = tokens.index(entity_tokens[0])
            labels[first_token_index] = f"B-{label}"
            for token in entity_tokens[1:]:
                labels[tokens.index(token)] = f"I-{label}"
    
    return tokens, labels

In [7]:
labels,sentences=[],[]
for i in range(len(data)):
    i,j=data[i]
    inp,lab=tokenize_and_label_nltk(i,j['entities'])
    sentences.append(inp)
    labels.append(lab)

In [8]:
max_len = 128
n_tags = len(set(tag for label in labels for tag in label)) + 1  # Add 1 for padding
word2idx = {w: i + 2 for i, w in enumerate(set(word for sent in sentences for word in sent))}
word2idx["PAD"] = 0
word2idx["UNK"] = 1
tag2idx = {t: i for i, t in enumerate(set(tag for label in labels for tag in label))}

In [9]:
train_sentences, val_sentences, train_labels, val_labels = train_test_split(sentences, labels, test_size=0.2)

In [10]:
class entityrecognitionpipeline:
    def __init__(self, vocab_size, max_len, entity_classes):
        self.tokenizer = Tokenizer(num_words=vocab_size, oov_token="<UNK>")
        self.max_len = max_len
        self.entity_classes = entity_classes
        self.label_to_id = {label: idx for idx, label in enumerate(entity_classes)}
        self.id_to_label = {idx: label for label, idx in self.label_to_id.items()}
    
    def fit_tokenizer(self, texts):
        self.tokenizer.fit_on_texts(texts)
    
    def encode_texts(self, texts):
        sequences = self.tokenizer.texts_to_sequences(texts)
        padded_sequences = pad_sequences(sequences, maxlen=self.max_len, padding='post', truncating='post')
        return padded_sequences
    
    def encode_labels(self, labels):
        encoded_labels = [[self.label_to_id[label] for label in seq] for seq in labels]
        padded_labels = pad_sequences(encoded_labels, maxlen=self.max_len, padding='post', truncating='post', value=self.label_to_id["O"])
        return padded_labels
    
    def create_mask(self, texts):
        sequences = self.tokenizer.texts_to_sequences(texts)
        mask = np.array([[1 if i < len(seq) else 0 for i in range(self.max_len)] for seq in sequences])
        return mask[:, np.newaxis, np.newaxis, :]  # Shape: (batch_size, 1, 1, max_len)
    
    def process_data(self, texts, labels):
        X = self.encode_texts(texts)
        y = self.encode_labels(labels)
        mask = self.create_mask(texts)
        return X, y, mask

In [11]:
VOCAB_SIZE = 50000
MAX_LEN = 128
ENTITY_CLASSES = set(tag for label in labels for tag in label)
MAX_LEN = 128
VOCAB_SIZE = 30522  
EMBED_DIM = 128  
NUM_HEADS = 8  
FF_DIM = 512  
NUM_LAYERS = 2  
NUM_CLASSES = len(ENTITY_CLASSES)  

In [12]:
ENTITY_CLASSES=[i for i in ENTITY_CLASSES]

In [14]:
pipeline = entityrecognitionpipeline(vocab_size=VOCAB_SIZE, max_len=MAX_LEN, entity_classes=ENTITY_CLASSES)
pipeline.fit_tokenizer(sentences)

In [15]:
X, y, mask = pipeline.process_data(train_sentences, train_labels)
test_X, test_y,test_mask = pipeline.process_data(train_sentences, train_labels)

In [16]:
def positional_enconding(length, depth):
    depth=depth/2
    #Dummy vectore  for encoding 
    positions= np.arange(length)[:, np.newaxis]
    depth=np.arange(depth)[np.newaxis, :]/depth
    # 1/10000^(i/d) for exis 0
    angle_rates= 1/(10000**depth)
    angle_rads=positions*angle_rates
    #concatinate  both exis
    pos_enconding=np.concatenate([np.sin(angle_rads), np.cos(angle_rads)], axis=-1)
    return tf.cast(pos_enconding,  dtype=tf.float32)

In [17]:
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.d_model=d_model
        self.embedding=tf.keras.layers.Embedding(vocab_size,d_model,mask_zero=True)
        # Calling positional enconding function
        self.pos_enconding=positional_enconding(length=1028, depth=d_model)
    
    def call(self, x):
        length=tf.shape(x)[1]
        x=self.embedding(x)
        x*=tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x=x+self.pos_enconding[np.newaxis, :length, :]
        return x

In [18]:
def scaled_dot_product_attention(q, k, v, mask=None):
    matmul_qk = tf.matmul(q, k, transpose_b=True)  # (..., seq_len_q, seq_len_k)

    dk = tf.cast(tf.shape(k)[-1], tf.float32)
    scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

    if mask is not None:
        scaled_attention_logits += (mask * -1e9)

    
    attention_weights =  tf.nn.softmax(scaled_attention_logits, axis=-1)

    output = tf.matmul(attention_weights, v)  # (..., seq_len_q, depth_v)
    return output, attention_weights


In [19]:
class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.embed_dim = embed_dim

        assert embed_dim % num_heads == 0

        self.depth = embed_dim // num_heads

        self.wq = Dense(embed_dim)
        self.wk = Dense(embed_dim)
        self.wv = Dense(embed_dim)

        self.dense = Dense(embed_dim)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, q, k, v, mask):
        batch_size = tf.shape(q)[0]

        q = self.split_heads(self.wq(q), batch_size)  # (batch_size, num_heads, seq_len_q, depth)
        k = self.split_heads(self.wk(k), batch_size)  # (batch_size, num_heads, seq_len_k, depth)
        v = self.split_heads(self.wv(v), batch_size)  # (batch_size, num_heads, seq_len_v, depth)

        scaled_attention, _ = scaled_dot_product_attention(q, k, v, mask)

        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])  # (batch_size, seq_len_q, num_heads, depth)

        concat_attention = tf.reshape(scaled_attention, (batch_size, -1, self.embed_dim))  # (batch_size, seq_len_q, embed_dim)

        output = self.dense(concat_attention)  # (batch_size, seq_len_q, embed_dim)

        return output

In [20]:
class TransformerEnc(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerEnc, self).__init__()
        self.att = MultiHeadAttention(embed_dim, num_heads)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training, mask):
        attn_output = self.att(inputs, inputs, inputs, mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

In [21]:
def build_model():
    inputs = Input(shape=(MAX_LEN,))
    mask = Input(shape=(1, 1, MAX_LEN))

    #x = Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, input_length=MAX_LEN)(inputs)
    emb=PositionalEmbedding(VOCAB_SIZE,EMBED_DIM)
    x=emb(inputs)
    for _ in range(NUM_LAYERS):
        x = TransformerEnc(embed_dim=EMBED_DIM, num_heads=NUM_HEADS, ff_dim=FF_DIM)(x, training=True, mask=mask)

    x = Dense(NUM_CLASSES, activation="softmax")(x)

    model = Model(inputs=[inputs, mask], outputs=x)
    return model

In [22]:
model = build_model()
model.compile(optimizer=Adam(learning_rate=0.001), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [23]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, 128, 128)  │  3,906,816 │ input_layer[0][0] │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 1, 1, 128) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_enc     │ (None, 128, 128)  │    198,272 │ positional_embed… │
│ (TransformerEnc)    │                   │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_enc_1   │ (None, 128, 128)  │    198,272 │ transformer_enc[… │
│ (TransformerEnc)    │                   │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 128, 23)   │      2,967 │ transformer_enc_… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,306,327 (16.43 MB)

 Trainable params: 4,306,327 (16.43 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
model.fit([X, mask], y, batch_size=32, epochs=10)

Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 9s 388ms/step - accuracy: 0.5736 - loss: 1.6307
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 366ms/step - accuracy: 0.8695 - loss: 0.7487
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 335ms/step - accuracy: 0.8735 - loss: 0.6743
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 348ms/step - accuracy: 0.8735 - loss: 0.6305
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 340ms/step - accuracy: 0.8773 - loss: 0.5681
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 339ms/step - accuracy: 0.8820 - loss: 0.5215
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 415ms/step - accuracy: 0.8954 - loss: 0.4362
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 344ms/step - accuracy: 0.9083 - loss: 0.3648
Epoch 9/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 343ms/step - accuracy: 0.9136 - loss: 0.3299
Epoch 10/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 343ms/step - accuracy: 0.9337 - loss: 0.2436


In [25]:
tokens=val_sentences[0]
sequence = [word2idx.get(w, word2idx["UNK"]) for w in tokens]
sequence = tf.keras.preprocessing.sequence.pad_sequences([sequence], maxlen=max_len, padding='post')

In [26]:
preds = model.predict([test_X,test_mask])

6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 170ms/step


In [27]:
preds

array([[[3.70259768e-05, 5.78295439e-04, 2.72744539e-04, ...,
         8.64475878e-05, 1.54795154e-04, 1.72389788e-04],
        [1.70610059e-04, 7.18327705e-04, 8.81943037e-04, ...,
         3.21113737e-04, 1.47827799e-04, 1.35032483e-03],
        [3.85219901e-04, 1.20918779e-03, 1.48915162e-03, ...,
         3.72361945e-04, 4.25541977e-04, 9.04567838e-01],
        ...,
        [2.14135696e-04, 2.28938716e-05, 7.31410455e-06, ...,
         1.02893850e-04, 8.54217433e-05, 6.00109852e-05],
        [6.99983910e-04, 1.91129584e-04, 3.74109150e-05, ...,
         1.55937116e-04, 1.80505929e-04, 5.80000749e-04],
        [3.30514013e-04, 8.75964543e-05, 1.17035779e-05, ...,
         4.41815610e-05, 7.44171266e-05, 2.61020119e-04]],

       [[1.59641349e-05, 2.37593893e-04, 1.44063873e-04, ...,
         3.57666468e-05, 8.28044576e-05, 1.17350166e-04],
        [4.57191636e-04, 2.84240174e-04, 1.52589835e-03, ...,
         1.70739615e-04, 6.58016361e-05, 1.73767854e-03],
        [2.09055003e-02, 

In [28]:
pred_tags = [list(tag2idx.keys())[list(tag2idx.values()).index(p)] for p in preds[3].argmax(axis=-1)]

In [29]:
list(zip(tokens, pred_tags))

[('Alok', 'B-Name'),
 ('Khandai', 'I-Name'),
 ('Operational', 'B-Designation'),
 ('Analyst', 'O'),
 ('(SQL', 'I-Designation'),
 ('DBA)', 'I-Designation'),
 ('Engineer', 'O'),
 ('-', 'I-Email Address'),
 ('UNISYS', 'B-Email Address'),
 ('Bengaluru,', 'O'),
 ('Karnataka', 'I-Degree'),
 ('-', 'O'),
 ('Email', 'I-Email Address'),
 ('me', 'I-Years of Experience'),
 ('on', 'O'),
 ('Indeed:', 'I-Location'),
 ('indeed.com/r/Alok-Khandai/5be849e443b8f467', 'O'),
 ('❖', 'O'),
 ('Having', 'O'),
 ('3.5', 'O'),
 ('Years', 'O'),
 ('of', 'O'),
 ('IT', 'B-Email Address'),
 ('experience', 'O'),
 ('in', 'O'),
 ('SQL', 'O'),
 ('Database', 'O'),
 ('Administration,', 'O'),
 ('System', 'O'),
 ('Analysis,', 'O'),
 ('Design,', 'O'),
 ('Development', 'O'),
 ('&', 'O'),
 ('Support', 'O'),
 ('of', 'O'),
 ('MS', 'O'),
 ('SQL', 'O'),
 ('Servers', 'O'),
 ('in', 'O'),
 ('Production,', 'O'),
 ('Development', 'O'),
 ('environments', 'O'),
 ('&', 'O'),
 ('Replication', 'O'),
 ('and', 'O'),
 ('Cluster', 'O'),
 ('Server'